In [1]:
import torch
import torch.nn as nn # TensorFlow의 layers 느낌 Dense = Linear 
import matplotlib.pyplot as plt

torch.manual_seed(42)



A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.5 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "c:\Users\Playdata\miniconda3\envs\test_env\lib\runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "c:\Users\Playdata\miniconda3\envs\test_env\lib\runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "c:\Users\Playdata\miniconda3\envs\test_env\lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "c:\Users\Playdata\miniconda3\envs\test_env\lib\site-packages\traitlets\config\application.py", line 1075, in launc

In [2]:
layer_no_act = []
for i in range(100):
    layer_no_act.append(nn.Linear(4,4))


deep_no_activation = nn.Sequential(*layer_no_act)

# 1층
single_layer = nn.Linear(4,4)

# 데이터
x_test = torch.randn(1,4)
# 초기값
w_ = torch.eye(4)
b_ = torch.zeros(4)

for layer in layer_no_act:
    w_ = layer.weight @ w_
    b_ = layer.weight @ b_ + layer.bias

output_ = x_test @ w_.T + b_ # wx+b

with torch.no_grad():
    output_100 = deep_no_activation(x_test)

output_100, output_

(tensor([[-0.1281,  0.0422, -0.2312, -0.4419]]),
 tensor([[-0.1281,  0.0422, -0.2312, -0.4419]], grad_fn=<AddBackward0>))

In [3]:
torch.manual_seed(42)
class SigmoidNet(nn.Module): # Module 상속
    # forward : 순전파 예측값 순차적으로 통과 시키는 것 
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(4,8)
        self.layer2 = nn.Linear(8,8)
        self.layer3 = nn.Linear(8,8)
        self.layer4 = nn.Linear(8,8)
        self.layer5 = nn.Linear(8,1)
    def forward(self, x):
        x = torch.sigmoid(self.layer1(x))
        x = torch.sigmoid(self.layer2(x))
        x = torch.sigmoid(self.layer3(x))
        x = torch.sigmoid(self.layer4(x))
        x = self.layer5(x)
        return x
    # 가장 간단한 파이토치의 모델 설계

sigmoid_net = SigmoidNet()
sigmoid_net

SigmoidNet(
  (layer1): Linear(in_features=4, out_features=8, bias=True)
  (layer2): Linear(in_features=8, out_features=8, bias=True)
  (layer3): Linear(in_features=8, out_features=8, bias=True)
  (layer4): Linear(in_features=8, out_features=8, bias=True)
  (layer5): Linear(in_features=8, out_features=1, bias=True)
)

In [7]:
x_input  = torch.randn(1,4) # 1차원 보다 2차원 형태를 요구 하기 때문에 처음레이어를 4,8로 설정했기 때문에
y_target = torch.tensor([[1.0]])
output = sigmoid_net(x_input) # 객체 자체의 값을 넣어주면 forward가 호출이 된다.
loss = (y_target - output)**2
loss.backward() # 각 계산마다 자동적으로 미분

In [15]:
print(f"{'레이어':>10} | {'레이어':>10} | {'레이어':>10}")

       레이어 |        레이어 |        레이어


In [25]:
for name, param in sigmoid_net.named_parameters():
    grad_magnitude =  param.grad.abs().mean().item() # 기울기들의 평균, item()을 하면 tensor 에서 빠져나옴 데이터를 가져올려고, tensor는 다른 세계
    status = "손실" if grad_magnitude < 0.001 else "보통" if grad_magnitude > 0.01 else "양호"
    print(name, grad_magnitude,status)

layer1.weight 0.0001970449520740658 손실
layer1.bias 0.0002025977155426517 손실
layer2.weight 0.0005189195508137345 손실
layer2.bias 0.0009850841015577316 손실
layer3.weight 0.0033919087145477533 양호
layer3.bias 0.007354467175900936 양호
layer4.weight 0.03504737839102745 보통
layer4.bias 0.07489149272441864 보통
layer5.weight 0.6412580013275146 보통
layer5.bias 1.4335606098175049 보통


In [28]:
torch.manual_seed(42)
import torch.nn as nn

class ReLUNet(nn.Module): # Module 상속
    # forward : 순전파 예측값 순차적으로 통과 시키는 것 
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(4,8)
        self.layer2 = nn.Linear(8,8)
        self.layer3 = nn.Linear(8,8)
        self.layer4 = nn.Linear(8,8)
        self.layer5 = nn.Linear(8,1)
        self.leakyRelu = nn.LeakyReLU()
    def forward(self, x):
        x = self.leakyRelu(self.layer1(x))
        x = self.leakyRelu(self.layer2(x))
        x = self.leakyRelu(self.layer3(x))
        x = self.leakyRelu(self.layer4(x))
        x = self.layer5(x)
        return x
    # 가장 간단한 파이토치의 모델 설계

relu_net = ReLUNet()
relu_net


x_input  = torch.randn(1,4) 
y_target = torch.tensor([[1.0]])
output = relu_net(x_input)
loss = (y_target - output)**2
loss.backward()

for name, param in relu_net.named_parameters():
    grad_magnitude =  param.grad.abs().mean().item() 
    status = "손실" if grad_magnitude < 0.001 else "보통" if grad_magnitude > 0.01 else "양호"
    print(name, grad_magnitude,status)

layer1.weight 0.003524485509842634 양호
layer1.bias 0.004581727087497711 양호
layer2.weight 0.003884008154273033 양호
layer2.bias 0.009393860585987568 양호
layer3.weight 0.0023075814824551344 양호
layer3.bias 0.02644430473446846 보통
layer4.weight 0.005813978612422943 양호
layer4.bias 0.06026383116841316 보통
layer5.weight 0.054165832698345184 보통
layer5.bias 1.8474719524383545 보통
